### Convolução usando Pytorch

Veremos em detalhe como realizar convolução usando Pytorch e como funciona a camada convolucional

In [9]:
import torch
import torch.nn.functional as F
from torch import nn

# Sinal
x = torch.tensor([5, 4, 8, 7, 9, 3, 6], dtype=torch.float32)
# Filtro
weight = torch.tensor([1, 2, 3], dtype=torch.float32)
# Tamanho do filtro
ks = len(weight)

# Redimensiona o sinal para o tamanho 1x1xlen(x). Ou seja, um batch contendo um único sinal, 
# e esse sinal possui um único canal
x = x.reshape(1,1,len(x))
# Redimensiona o filtro. O primeiro valor 1 possui um significado diferente do que
# no caso do sinal. Depois será explicado.
weight = weight.reshape(1,1,len(weight))
# Realiza a convolução
y = F.conv1d(x, weight)

print(y)
print(y.shape)

tensor([[[37., 41., 49., 34., 33.]]])
torch.Size([1, 1, 5])


O tamanho da entrada é 7 e da saída é 5. Isso porque o Pytorch realiza a convolução apenas nas posições que não necessitam de preenchimento de borda. Mas modificar o tamanho do resultado é indesejável. 

É muito comum realizarmos a convolução com padding para manter o tamanho do sinal:

In [10]:
# padding = ks//2 garante que a saída sempre terá o mesmo tamanho que a entrada
y = F.conv1d(x, weight, padding=ks//2)
print(y.shape)

torch.Size([1, 1, 7])


Dado o sinal [5, 4, 8, 7, 9, 3, 6] e filtro [1,2,3], nossa saída deve ser:

* y[0] = 1\*0 + 2\*5 + 3\*4 = 22
* y[1] = 1\*5 + 2\*4 + 3\*8 = 37
* ...
* y[6] = 1\*3 + 2\*6 + 3\*0 = 15

Note que a função realiza a correlação-cruzada, e não a convolução. Mas para redes neurais isso não importa.

In [11]:
print(y)

tensor([[[22., 37., 41., 49., 34., 33., 15.]]])


Em redes neurais a convolução possui o conceito de bias, que é simplesmente um valor constante que é adicionado ao resultado:

In [12]:
bias = torch.tensor([5.])
# Adiciona o valor 5 a cada elemento do resultado da convolução
y_bias = F.conv1d(x, weight, padding=ks//2, bias=bias)
print(y_bias)
print(torch.allclose(y+bias, y_bias))

tensor([[[27., 42., 46., 54., 39., 38., 20.]]])
True


### Camada de convolução

In [13]:
# A entrada terá 1 canal, queremos apenas 1 canal de saída. O tamanho do filtro é ks, o 
# padding é metade do tamanho do filtro e a camada não terá bias
conv = nn.Conv1d(in_channels=1, out_channels=1, kernel_size=ks, padding=ks//2, bias=False)
y = conv(x)
print(y)

tensor([[[2.2447, 2.9537, 3.1152, 3.1424, 0.9340, 1.2492, 0.0635]]],
       grad_fn=<ConvolutionBackward0>)


Uma camada de convolução consiste em um filtro possuindo valores aleatórios. Esse filtro possui o parâmetro requires_grad=True por padrão. Podemos alterar os valores do filtro se quisermos:

In [14]:
print(conv.weight.shape)
print(conv.weight)

with torch.no_grad():
    conv.weight[:] = weight

# Mesmo resultado que a convolução que fizemos antes:
conv(x)

torch.Size([1, 1, 3])
Parameter containing:
tensor([[[-0.1840,  0.1026,  0.4329]]], requires_grad=True)


tensor([[[22., 37., 41., 49., 34., 33., 15.]]], grad_fn=<ConvolutionBackward0>)

### Relação entre convolução e combinação linear

Uma convolução nada mais é do que uma combinação linear com menos parâmetros

In [15]:
# Matriz de convolução. Cada linha representa uma posição do kernel. Por exemplo,
# a linha 0 fará a operação 2*x[0]+3*x[1]+0*x[2]+...
matrix = torch.tensor([[2, 3, 0, 0, 0, 0, 0],
                       [1, 2, 3, 0, 0, 0, 0],
                       [0, 1, 2, 3, 0, 0, 0],
                       [0, 0, 1, 2, 3, 0, 0],
                       [0, 0, 0, 1, 2, 3, 0],
                       [0, 0, 0, 0, 1, 2, 3],
                       [0, 0, 0, 0, 0, 1, 2]], dtype=torch.float32)

F.linear(x, matrix)

tensor([[[22., 37., 41., 49., 34., 33., 15.]]])

Acima temos uma matrix 7x7 que recebe 7 atributos de entrada e gera 7 atributos de saída. Mas a combinação linear dos atributos de entrada sempre envolvem apenas 3 parâmetros.

### Convolução com mais de um canal

In [16]:
# Batch contendo um sinal de tamanho 7 e com 4 canais
x = torch.rand(size=(1,4,7))
# Camada que recebe sinal com 4 canais e gera um sinal com 5 canais.
conv = nn.Conv1d(in_channels=4, out_channels=5, kernel_size=ks, padding=ks//2, bias=False)

y = conv(x)
print("x\n",x)
# Saída possui tamanho 1x5x7
print("y\n",y)

x
 tensor([[[0.3161, 0.8651, 0.9076, 0.2607, 0.6921, 0.7542, 0.1646],
         [0.0669, 0.6400, 0.7325, 0.1067, 0.0129, 0.6392, 0.5094],
         [0.9757, 0.9243, 0.6445, 0.8684, 0.6136, 0.6987, 0.8517],
         [0.8007, 0.0272, 0.5845, 0.0862, 0.4311, 0.6522, 0.0074]]])
y
 tensor([[[ 0.3406,  0.3128,  0.3343, -0.0019,  0.1962,  0.3131,  0.0670],
         [-0.3243, -0.1148,  0.1020, -0.0019, -0.2195, -0.2250,  0.0810],
         [-0.0570,  0.3915,  0.1340, -0.1724,  0.1898,  0.1687, -0.0381],
         [ 0.1419, -0.0507,  0.2030, -0.0097,  0.0964,  0.2212,  0.1181],
         [ 0.3300,  0.3790,  0.0871,  0.2377,  0.2704,  0.2336,  0.1975]]],
       grad_fn=<ConvolutionBackward0>)


* O **número de canais de saída** define o **número de filtros** que serão utilizados na camada de convolução. No nosso caso, temos 5 filtros
* Cada filtro possui **tamanho espacial ks**
* Cada filtro possui **profundidade 4**, pois o sinal de entrada possui 4 canais.
* Portanto, temos 5 filtros de tamanho 4 x ks cada
* Portanto, o tamanho do tensor .weight da camada de convolução possui tamanho 5 x 4 x ks

In [17]:
conv.weight.shape

torch.Size([5, 4, 3])

In [18]:
# Filtro 1 da camada
filtro1 = conv.weight[0]
# Região do sinal que corresponde quando o filtro está na posição 1
regiao = x[0,:,0:3]
# Resultado da convolução para esse ponto específico
res = (filtro1*regiao).sum()
# comparação do resultado
print(torch.allclose(y[0,0,1], res))

True


### Outros parâmetros da convolução

#### Stride
Stride define a quantidade de deslocamento do filtro para cada posição na qual a convolução será calculada

In [19]:
x = torch.tensor([5, 4, 8, 7, 9, 3, 6], dtype=torch.float32)
weight = torch.tensor([1, 2, 3], dtype=torch.float32)
x = x.reshape(1,1,len(x))
weight = weight.reshape(1,1,len(weight))

# Faz o filtro deslocar duas posições ao invés de deslocar uma posição por vez
y = F.conv1d(x, weight, stride=2)
print(y)

tensor([[[37., 49., 33.]]])


In [20]:
# O resultado de stride=2 é equivalente a indexar a saída de stride=1 pulando 2 índices
yt = F.conv1d(x, weight, stride=1)
print(torch.allclose(y, yt[0,0,::2]))

True


#### Dilatação

Dilatação consiste em aumentar o tamanho do filtro sem aumentar o número de parâmetros

In [21]:
y = F.conv1d(x, weight, dilation=2)
print(y)

tensor([[[48., 27., 44.]]])


In [22]:
# dilatação=2 é equivalente a inserir 0 entre os valores do filtro
wt = torch.tensor([1, 0, 2, 0, 3], dtype=torch.float32).reshape(1,1,-1)
yt = F.conv1d(x, wt, dilation=1)
print(torch.allclose(y, yt))

True
